# B2-rand — tiếp tục validation từ output mới nhất

Add Input `grace_b2_rand_inputs.zip`, GGUF Q4_K_M đã pin, và **checkpoint mới nhất** `grace_b2_rand_checkpoint.zip` từ phiên I-12. Output đã kiểm tra tại `Runs-management/kaggle_outputs_20261107/b2_rand_latest/`; SHA-256 checkpoint: `5a1e049fe807fff413129bfb25e3977b55bdf8b4abd35b078aa373a10dbff197`.

I-12 đã đủ validation cho seed 42 và 43 (mỗi seed 5.005/5.005). Seed 44 có Devign 2.426/2.732; ReVeal 0/2.273. Tổng lũy kế **12.436/15.015**, còn **2.579**: seed 44 Devign 306 và ReVeal 2.273.

Notebook tự nhận checkpoint I-12 hoặc hậu duệ có nguyên prefix I-11, bỏ qua ID đã hoàn tất và chỉ chạy validation. Giới hạn 9 giờ gồm cả setup/model load. Khi hết phiên, tải checkpoint mới từ Output và dùng nó để resume. Notebook dừng khi validation đủ; không tự fit ngưỡng và không chạy test.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone
import threading, signal

# Re-running setup in the same session must not restart the time budget.
if "SESSION_STARTED_EPOCH" not in globals(): SESSION_STARTED_EPOCH=time.time()
SESSION_DEADLINE_EPOCH=SESSION_STARTED_EPOCH+9*3600

DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log đầy đủ: {log}')
    is_inference = name.startswith('resume_seed_')
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env, start_new_session=True)
        def stop_process_group():
            if proc.poll() is not None: return
            note(f'Watchdog: stopping {name}; preserving time for checkpoint export')
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError: pass
        watchdog=threading.Timer(max(0,SESSION_DEADLINE_EPOCH-300-time.time()),stop_process_group)
        watchdog.daemon=True
        watchdog.start()
        for line in proc.stdout:
            f.write(line); f.flush()
            if re.search(r'\b(?:Found|Skip(?:ping)?) link\b', line, flags=re.IGNORECASE):
                continue
            if is_inference:
                match = re.search(r'^(\S+).*\((\d+)/(\d+)\)\s*$', line.strip())
                if match:
                    run_key, done, total = match.group(1), int(match.group(2)), int(match.group(3))
                    if done % 100 == 0 or done == total:
                        print(f'Inference {run_key}: {done}/{total}', flush=True)
                elif line.strip():
                    print(line, end='', flush=True)
            else:
                print(line, end='', flush=True)
        code = proc.wait()
        watchdog.cancel()
    if is_inference and '--status' in cmd:
        status_path = Path(cmd[cmd.index('--status') + 1])
        if status_path.is_file():
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print(f"Inference summary: {status.get('completed_records_total', 0)} records; state={status.get('state')}", flush=True)
            for run_key, progress in status.get('runs', {}).items():
                print(f"  {run_key}: {progress.get('completed', 0)}/{progress.get('count', '?')} [{progress.get('state', 'pending')}]")
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions():
    result = {}
    for name in ('llama-cpp-python', 'transformers', 'huggingface_hub', 'scikit-learn', 'numpy', 'scipy'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = 'not installed'
    note('Installed package versions: ' + json.dumps(result))
    return result

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace = ''.join(traceback.format_exception(etype, evalue, tb))
    try:
        with (DIAG / 'exceptions.log').open('a', encoding='utf-8') as f:
            f.write(f'\n[{datetime.now(timezone.utc).isoformat()}]\n{trace}\n')
        note(f'Lỗi: {etype.__name__}: {evalue}')
        versions()
        diagnostics_zip()
    except Exception as diagnostic_error:
        print('Không thể ghi đầy đủ chẩn đoán:', diagnostic_error, flush=True)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,), capture_failure)
note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Executable: ' + sys.executable)
note('Platform: ' + platform.platform())
for loc in ('/kaggle/working', '/kaggle/input'):
    try:
        usage = shutil.disk_usage(loc)
        note(f'Disk {loc}: total={usage.total} free={usage.free} bytes')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions()
for name, cmd in [('gpu', ['nvidia-smi']), ('cuda_compiler', ['nvcc', '--version']), ('cmake', ['cmake', '--version']), ('pip', [sys.executable, '-m', 'pip', '--version'])]:
    try: run_logged(cmd, name)
    except (OSError, subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
diagnostics_zip()


## Chọn bundle và cấu hình chạy


In [ ]:
# Gắn đúng Kaggle Dataset bundle và Notebook Output chứa model.
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
BUNDLE_WORK = Path('/kaggle/working/grace_b2_rand_bundles')
MODEL_FILENAME = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335'
SEEDS = (42, 43, 44)
# Bắt buộc tiếp tục từ I-11 hoặc bản mới hơn chứa nguyên lịch sử I-09/I-11.
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_INPUT_PATH = None  # Có thể đặt Path tới ZIP hoặc thư mục đã giải nén để chọn cụ thể.
MODEL_REVISION = None  # None: khóa revision Hub tại thời điểm tải
LIMIT = None           # Chạy toàn bộ từng split
MAX_RUNTIME_HOURS = 9
RUN_DEADLINE_EPOCH = SESSION_STARTED_EPOCH + MAX_RUNTIME_HOURS * 3600
CHECKPOINT_RESERVE_SECONDS = 600  # Dừng tạo record mới trước hạn để đóng gói checkpoint
GPU_LAYERS = -1
OUT_ROOT = Path('/kaggle/working/grace_b2_rand_three_seed_11h')
EXPECTED_RUNS = [('devign', 'validation'), ('reveal', 'validation')]


In [ ]:
# Đọc bundle lịch sử; chỉ materialize 6 tập validation; xác minh arm, seed, hash, ID và số mẫu.
import io
BUNDLE_WORK.mkdir(parents=True, exist_ok=True)
found = {}
def register_bundle(manifest_bytes, prompt_bytes, origin):
    manifest = json.loads(manifest_bytes)
    if manifest.get('arm') != 'B2-rand': return
    key = (manifest.get('seed'), manifest.get('dataset'), manifest.get('split'))
    if key[0] not in SEEDS or key[1:] not in EXPECTED_RUNS: return
    digest = hashlib.sha256(prompt_bytes).hexdigest()
    if digest != manifest['prompts_sha256']: raise ValueError(f'Hash sai: {origin}')
    rows = [json.loads(x) for x in prompt_bytes.decode('utf-8').splitlines() if x.strip()]
    if len(rows) != manifest['count'] or len({str(x['id']) for x in rows}) != len(rows): raise ValueError(f'Count/ID sai: {origin}')
    if any((x.get('seed'),x.get('dataset'),x.get('split'),x.get('arm')) != (*key,'B2-rand') or hashlib.sha256(x['prompt'].encode()).hexdigest()!=x['prompt_sha256'] for x in rows): raise ValueError(f'Identity/prompt hash sai: {origin}')
    if key in found and found[key]['digest'] != digest: raise ValueError(f'Bundle trùng khác nội dung: {key}')
    found[key] = dict(manifest=manifest_bytes,prompts=prompt_bytes,digest=digest,origin=str(origin))
for path in sorted(KAGGLE_INPUT_ROOT.rglob('bundle_manifest.json')):
    prompts = path.with_name('prompts.jsonl')
    if prompts.is_file(): register_bundle(path.read_bytes(),prompts.read_bytes(),path.parent)
archives = sorted(set(KAGGLE_INPUT_ROOT.rglob('grace_b2_rand_inputs.zip')) | set(KAGGLE_INPUT_ROOT.rglob('*_b2_rand_seed_*_kaggle.zip')))
for path in archives:
    with zipfile.ZipFile(path) as z:
        names=set(z.namelist())
        if {'bundle_manifest.json','prompts.jsonl'} <= names: register_bundle(z.read('bundle_manifest.json'),z.read('prompts.jsonl'),path)
        for member in sorted(x for x in names if '_b2_rand_seed_' in x and x.endswith('_kaggle.zip')):
            with zipfile.ZipFile(io.BytesIO(z.read(member))) as inner:
                register_bundle(inner.read('bundle_manifest.json'),inner.read('prompts.jsonl'),f'{path}!{member}')
expected=[(seed,ds,sp) for seed in SEEDS for ds,sp in EXPECTED_RUNS]
missing=[key for key in expected if key not in found]
if missing: raise FileNotFoundError(f'Thiếu bundle: {missing}. Add Input grace_b2_rand_inputs.zip')
# Hash của đúng 6 tập validation đã audit; không chấp nhận bundle mới chỉ vì manifest tự khớp.
AUDITED_PROMPT_HASHES = {(seed, ds, sp): digest for seed,ds,sp,digest in [[42,"devign","validation","8112a6478f8eecd8e7011af881ce9057d02caf42cb8367515108785ff55ad65a"],[43,"devign","validation","1e9b6fc3d3e5e129ecf1e779b959056de0f9b3b5b8386b366b74443c9f383f24"],[44,"devign","validation","fe0c739a29a591cf75b957ef2e86c53fb7ed6b83ba9c4fd29509d76490d10c7e"],[42,"reveal","validation","dd08b1ffd36690bf6f2ab00f8fda1708d6b88e6fe9957893519075d35d06d8a3"],[43,"reveal","validation","c4d6550b0432ba4e347d2444fe39648054c4616a49fa0d5c6a11f50f1e5a4ad6"],[44,"reveal","validation","8401afc0a212b45d5eaed4bf1a74919414b0e9683699b9f0d268432341ae81d1"],[42,"devign","test","237a138fc911224fe8d0059bc101971f33d8a1e182e7be3d911535d2a3fcb7cb"],[43,"devign","test","6bcd35410960ce4de0f6b9ae048d7fc5ee295ed93baa995b26754cd0f5561f98"],[44,"devign","test","b3ef2ef95ccb5d99b739ed6fbf30aa876f170f8d2def26d48ebe3eece81261d9"],[42,"reveal","test","39ccb2c61797c272a4dcca0419887587e3197bcc7869497ad6a52e8d19c93a66"],[43,"reveal","test","a1ff4f3295b70a477150b9e118f7bef3abfe8edacb557222e1efd9fcdcc518d9"],[44,"reveal","test","8e35d22c5d1413d67200730fec4ce58270d2d7250a84f572394d024ffe825785"]]}
for key in expected:
    if found[key]['digest'] != AUDITED_PROMPT_HASHES[key]:
        raise ValueError(f'Prompt bundle thay đổi so với bản đã audit: {key}')
note('6 tập validation khớp hash đã audit trước phiên resume')
BUNDLE_PATHS={}
for key in expected:
    seed,ds,sp=key; item=found[key]; dest=BUNDLE_WORK/f'{seed}_{ds}_{sp}.zip'
    with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr('bundle_manifest.json',item['manifest']); z.writestr('prompts.jsonl',item['prompts'])
    BUNDLE_PATHS[key]=dest
    note(f'Bundle {key}: {json.loads(item["manifest"])["count"]} mẫu, sha256={item["digest"]}, source={item["origin"]}')
diagnostics_zip()



## Cài thư viện và xác minh CUDA


In [ ]:
# Cài từng nhóm để nhận diện chính xác bước thất bại; giữ nguyên phiên bản baseline.
# T4 dùng SM75; Kaggle đặt libcuda.so trong /usr/local/nvidia/lib64 nên thêm thư mục này cho CMake.
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64'
os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
os.environ['FORCE_CMAKE'] = '1'
note('Build config: CMAKE_ARGS=' + os.environ['CMAKE_ARGS'] + '; CMAKE_BUILD_PARALLEL_LEVEL=' + os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] + '; FORCE_CMAKE=' + os.environ['FORCE_CMAKE'])
run_logged([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'numpy==2.1.3', 'transformers==4.57.1', 'huggingface_hub==0.36.2', 'tokenizers==0.22.2', 'scikit-learn==1.5.2'], 'install_dependencies')
run_logged([sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-binary', 'llama-cpp-python', 'llama-cpp-python==0.3.16'], 'install_llama_cuda')
versions()
diagnostics_zip()


In [ ]:
import llama_cpp
versions()
note('llama-cpp-python GPU offload: ' + str(llama_cpp.llama_supports_gpu_offload()))
assert llama_cpp.llama_supports_gpu_offload(), 'llama-cpp-python chưa có CUDA; xem install_llama_cuda.log'
diagnostics_zip()


## Tải model và chạy inference


In [ ]:
# Tìm đúng tên file model trong toàn bộ Input đã gắn, bất kể username/slug.
model_files = sorted(path for path in KAGGLE_INPUT_ROOT.rglob(MODEL_FILENAME) if path.is_file())
if len(model_files) != 1:
    raise FileNotFoundError(f'Cần đúng một {MODEL_FILENAME} trong Kaggle Input; '
                            f'tìm thấy: {list(map(str, model_files))}. '
                            'Hãy Add Input chứa model và bỏ bản trùng nếu có; '
                            'notebook không tự tải lại GGUF từ Hugging Face.')
model_path = str(model_files[0])
revision = 'local-input'
note(f'Dùng model input: {model_path}')
print('Model input:', model_path)
print('Batch output directory:', OUT_ROOT)


In [ ]:
%%writefile /kaggle/working/kaggle_inference.py
import argparse
import hashlib
import json
import math
import os
import platform
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json_atomic(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    os.replace(tmp, path)


def binary_scores(logits, label_ids):
    values = np.asarray(logits, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    maximum = np.max(values)
    normalizer = float(maximum + np.log(np.exp(values - maximum).sum()))
    lp0 = float(values[label_ids[0]] - normalizer)
    lp1 = float(values[label_ids[1]] - normalizer)
    score = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))
    best = int(np.argmax(values))
    top = np.argsort(values)[-5:][::-1]
    return {'logprob_0': lp0, 'logprob_1': lp1, 'p0': math.exp(lp0), 'p1': math.exp(lp1),
            's': score, 'pred_argmax': int(score >= 0.5), 'greedy_token_id': best,
            'top5': [{'token_id': int(i), 'logprob': float(values[i] - normalizer)} for i in top]}


def load_bundle(bundle):
    with zipfile.ZipFile(bundle) as z:
        manifest = json.loads(z.read('bundle_manifest.json'))
        prompt_bytes = z.read('prompts.jsonl')
    if hashlib.sha256(prompt_bytes).hexdigest() != manifest['prompts_sha256']:
        raise ValueError(f'Prompt bundle hash mismatch: {bundle}')
    rows = [json.loads(line) for line in prompt_bytes.decode('utf-8').splitlines() if line.strip()]
    if len(rows) != manifest['count'] or len({str(r['id']) for r in rows}) != len(rows):
        raise ValueError(f'Prompt count or unique ID check failed: {bundle}')
    return manifest, rows


def evaluate_output(output):
    from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                                 confusion_matrix, f1_score, matthews_corrcoef, precision_score,
                                 recall_score, roc_auc_score)
    records = [json.loads(line) for line in output.read_text(encoding='utf-8').splitlines() if line.strip()]
    actual = [r for r in records if r['response_valid']]
    truth = [r['label'] for r in actual]
    predicted = [int(r['raw']) for r in actual]
    metrics = {'n_total': len(records), 'n_score_valid': sum(bool(r['valid']) for r in records),
               'n_response_valid': len(actual),
               'response_coverage': len(actual) / len(records) if records else None,
               'accuracy': None, 'balanced_accuracy': None, 'precision': None, 'recall': None,
               'f1': None, 'mcc': None, 'confusion_matrix': None, 'auroc': None, 'auprc': None}
    if actual:
        metrics.update(accuracy=accuracy_score(truth, predicted),
                       balanced_accuracy=balanced_accuracy_score(truth, predicted),
                       precision=precision_score(truth, predicted, zero_division=0),
                       recall=recall_score(truth, predicted, zero_division=0),
                       f1=f1_score(truth, predicted, zero_division=0),
                       mcc=matthews_corrcoef(truth, predicted),
                       confusion_matrix=confusion_matrix(truth, predicted, labels=[0, 1]).tolist())
    score_rows = [r for r in records if r['valid']]
    if score_rows and len({r['label'] for r in score_rows}) == 2:
        labels = [r['label'] for r in score_rows]
        scores = [r['s'] for r in score_rows]
        metrics.update(auroc=roc_auc_score(labels, scores),
                       auprc=average_precision_score(labels, scores))
    return metrics


def run_batch(runs, model_path, model_revision, gpu_layers, limit, deadline_epoch,
              checkpoint_reserve_seconds, status_path, seed, expected_model_sha256, arm):
    # Hard gate: a direct CLI invocation must also refuse test before model loading.
    if arm != 'B2-rand' or any(split != 'validation' for _,split,_,_ in runs):
        raise ValueError('Validation-only runner: test is blocked; stop and lock thresholds separately')
    from llama_cpp import Llama, __version__ as llama_version
    from transformers import AutoTokenizer
    if llama_version != '0.3.16': raise ValueError(f'Backend version mismatch: {llama_version}')

    started = time.time()
    stop_at = deadline_epoch - checkpoint_reserve_seconds
    status_path = Path(status_path)
    status_path.parent.mkdir(parents=True, exist_ok=True)
    status = {'state': 'starting', 'timed_out': False,
              'started_at': datetime.fromtimestamp(started, timezone.utc).isoformat(),
              'deadline_at': datetime.fromtimestamp(deadline_epoch, timezone.utc).isoformat(),
              'stop_new_records_at': datetime.fromtimestamp(stop_at, timezone.utc).isoformat(),
              'max_runtime_hours': (deadline_epoch - started) / 3600,
              'checkpoint_reserve_seconds': checkpoint_reserve_seconds,
              'runs': {}, 'completed_records_total': 0}
    for dataset, split, bundle, out in runs:
        key = f'{dataset}/{split}'
        status['runs'][key] = {'state': 'pending', 'bundle': bundle, 'output': out}
    write_json_atomic(status_path, status)

    def stop_for_deadline(current=None):
        status['state'] = 'timed_out'
        status['timed_out'] = True
        status['timed_out_at'] = datetime.now(timezone.utc).isoformat()
        status['stopped_after_run'] = current
        write_json_atomic(status_path, status)

    if time.time() >= stop_at:
        stop_for_deadline(None)
        return

    model_path = Path(model_path)
    model_digest = sha256(model_path)
    if model_digest != expected_model_sha256: raise ValueError('Model SHA256 không khớp contract')
    if model_revision == 'local-input':
        model_revision = f'local-sha256:{model_digest}'
    first_manifest, _ = load_bundle(runs[0][2])
    for dataset, split, bundle, out in runs:
        current_manifest, _ = load_bundle(bundle)
        if (current_manifest['tokenizer'], current_manifest['tokenizer_revision'],
            current_manifest['prompt_budget_tokens']) != (first_manifest['tokenizer'],
            first_manifest['tokenizer_revision'], first_manifest['prompt_budget_tokens']):
            raise ValueError('All four bundles must use the same tokenizer revision and context budget')
    tokenizer = AutoTokenizer.from_pretrained(first_manifest['tokenizer'],
                                                revision=first_manifest['tokenizer_revision'])
    llm = Llama(model_path=str(model_path), n_ctx=first_manifest['prompt_budget_tokens'], n_batch=256,
                n_gpu_layers=gpu_layers, logits_all=True, verbose=True)
    if time.time() >= stop_at:
        stop_for_deadline(None)
        return
    label_ids = []
    for label in ('0', '1'):
        hf_ids = tokenizer.encode(label, add_special_tokens=False)
        gguf_ids = llm.tokenize(label.encode(), add_bos=False, special=False)
        if len(hf_ids) != 1 or hf_ids != gguf_ids:
            raise ValueError(f'Label token mismatch for {label}: HF={hf_ids}, GGUF={gguf_ids}')
        label_ids.append(hf_ids[0])
    if label_ids[0] == label_ids[1]:
        raise ValueError('Label token IDs must differ')

    status['state'] = 'running'
    write_json_atomic(status_path, status)
    for dataset, split, bundle, out_text in runs:
        key = f'{dataset}/{split}'
        out = Path(out_text)
        out.mkdir(parents=True, exist_ok=True)
        manifest, rows = load_bundle(bundle)
        if (manifest.get('arm'), manifest.get('seed'), manifest['dataset'], manifest['split']) != (arm, seed, dataset, split):
            raise ValueError(f'Bundle identity mismatch for {key}: {manifest["dataset"]}/{manifest["split"]}')
        if limit is not None:
            rows = rows[:limit]
        identity = {'bundle_sha256': sha256(bundle),
                    'prompt_set_sha256': manifest['prompts_sha256'], 'model_sha256': model_digest,
                    'model_repo': manifest['model_repo'], 'quantization': manifest['quantization'],
                    'model_file': model_path.name, 'llama_cpp_python': llama_version,
                    'model_repo_revision': model_revision, 'tokenizer': manifest['tokenizer'],
                    'tokenizer_revision': manifest['tokenizer_revision'], 'gpu_layers': gpu_layers,
                    'context_tokens': manifest['prompt_budget_tokens'],
                    'scoring': 'unconstrained_first_token_logits_v1', 'limit': limit,
                    'python': platform.python_version(), 'dataset': dataset, 'split': split, 'arm': arm, 'prompt_seed': seed}
        identity_file = out / 'run_manifest.json'
        previous_identity = (json.loads(identity_file.read_text(encoding='utf-8'))
                             if identity_file.exists() else None)
        if previous_identity is not None:
            # Ignore only archive-byte hashes: ZIP timestamps/compression metadata may change.
            # Prompt identity is verified below against every checkpoint row's prompt_sha256.
            ignored = {'bundle_sha256', 'prompt_set_sha256'}
            previous_stable = {k: v for k, v in previous_identity.items() if k not in ignored}
            current_stable = {k: v for k, v in identity.items() if k not in ignored}
            if previous_stable != current_stable:
                raise ValueError(f'Existing output has different model/run settings for {key}; use a new OUT_ROOT')
            if ('prompt_set_sha256' in previous_identity and
                    previous_identity['prompt_set_sha256'] != identity['prompt_set_sha256']):
                raise ValueError(f'Prompt bundle content changed for {key}; refusing to resume')
        output = out / 'records.jsonl'
        done = {}
        if output.exists():
            for line in output.read_text(encoding='utf-8').splitlines():
                if line.strip():
                    row = json.loads(line)
                    row_id = str(row['id'])
                    if row_id in done:
                        raise ValueError(f'Duplicate checkpoint ID in {key}: {row_id}')
                    done[row_id] = row
        expected_ids = {str(item['id']) for item in rows}
        unexpected_ids = set(done) - expected_ids
        if unexpected_ids:
            raise ValueError(f'Checkpoint contains IDs absent from current bundle {key}: {sorted(unexpected_ids)[:10]}')
        for item in rows:
            row_id = str(item['id'])
            if row_id in done:
                if done[row_id].get('prompt_sha256') != item['prompt_sha256']:
                    raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                if done[row_id].get('model_sha256') != model_digest:
                    raise ValueError(f'Checkpoint model hash mismatch: {key}/{row_id}')
        # Persist the refreshed archive hash only after checkpoint IDs and prompt hashes pass.
        identity_file.write_text(json.dumps(identity, indent=2), encoding='utf-8')
        progress = status['runs'][key]
        progress.update(state='running', count=len(rows), completed=len(done),
                        records_path=str(output), bundle_sha256=identity['bundle_sha256'])
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)
        if len(done) > len(rows):
            raise ValueError(f'Checkpoint has more rows than this bundle: {key}')
        print(f'{key}: verified checkpoint IDs={len(done)}; skipping these; remaining={len(rows) - len(done)}', flush=True)

        with output.open('a', encoding='utf-8', newline='\n') as f:
            for item in rows:
                row_id = str(item['id'])
                if row_id in done:
                    if done[row_id]['prompt_sha256'] != item['prompt_sha256']:
                        raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                    continue
                if time.time() >= stop_at:
                    progress.update(state='checkpointed', completed=len(done))
                    status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                    stop_for_deadline(key)
                    return
                prompt = item['prompt']
                if hashlib.sha256(prompt.encode()).hexdigest() != item['prompt_sha256']:
                    raise ValueError(f'Prompt hash mismatch: {key}/{row_id}')
                messages = [{'role': 'user', 'content': prompt}]
                chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                hf_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
                gguf_ids = llm.tokenize(chat.encode('utf-8'), add_bos=False, special=True)
                if list(hf_ids) != gguf_ids:
                    raise ValueError(f'Tokenizer mismatch at {key}/{row_id}; abort rather than alter prompt')
                if len(gguf_ids) > manifest['prompt_budget_tokens']:
                    raise ValueError(f'Token budget exceeded at {key}/{row_id}')
                start = time.monotonic()
                llm.reset()
                llm.eval(gguf_ids)
                scores = binary_scores(llm.scores[llm.n_tokens - 1], label_ids)
                raw = llm.detokenize([scores['greedy_token_id']]).decode('utf-8', errors='replace').strip()
                row = {k: item[k] for k in ('id', 'dataset', 'split', 'label', 'prompt_sha256', 'demo_ids', 'tokens', 'graph_missing')}
                row.update(scores, arm=arm, raw=raw, response_valid=raw in ('0', '1'), valid=True,
                           token_ids={'0': label_ids[0], '1': label_ids[1]}, model_sha256=model_digest,
                           backend='llama-cpp-python', quantization='Q4_K_M', model_repo=manifest['model_repo'],
                           model_repo_revision=model_revision, seed=seed, timestamp=datetime.now(timezone.utc).isoformat(),
                           logprob_distribution='unconstrained_first_token', elapsed_s=time.monotonic()-start)
                f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
                f.flush()
                os.fsync(f.fileno())
                done[row_id] = row
                progress['completed'] = len(done)
                status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                status['last_checkpoint_at'] = datetime.now(timezone.utc).isoformat()
                write_json_atomic(status_path, status)
                print(f'{key} {row_id}: s={scores["s"]:.4f}, raw={raw!r} ({len(done)}/{len(rows)})', flush=True)
                if time.time() >= stop_at:
                    progress['state'] = 'checkpointed'
                    stop_for_deadline(key)
                    return

        metrics = evaluate_output(output)
        (out / 'evaluation.json').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')
        progress.update(state='complete', completed=len(rows), evaluation=str(out / 'evaluation.json'))
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)

    status['state'] = 'complete'
    status['timed_out'] = False
    status['completed_at'] = datetime.now(timezone.utc).isoformat()
    write_json_atomic(status_path, status)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True)
    ap.add_argument('--seed', type=int, required=True)
    ap.add_argument('--arm', choices=['B2-rand'], required=True)
    ap.add_argument('--expected-model-sha256', required=True)
    ap.add_argument('--model-revision', required=True)
    ap.add_argument('--gpu-layers', type=int, default=-1)
    ap.add_argument('--limit', type=int)
    ap.add_argument('--deadline-epoch', type=float, required=True)
    ap.add_argument('--checkpoint-reserve-seconds', type=int, default=300)
    ap.add_argument('--status', required=True)
    ap.add_argument('--run', action='append', nargs=4, metavar=('DATASET', 'SPLIT', 'BUNDLE', 'OUT'), required=True)
    args = ap.parse_args()
    run_batch(args.run, args.model, args.model_revision, args.gpu_layers, args.limit,
              args.deadline_epoch, args.checkpoint_reserve_seconds, args.status, args.seed, args.expected_model_sha256, args.arm)


In [ ]:
# Khôi phục checkpoint I-11 hoặc descendant; xác minh đầy đủ lịch sử I-09 và I-11.
from pathlib import PurePosixPath

def zip_files(path):
    with zipfile.ZipFile(path) as z:
        entries = z.infolist(); names = [x.filename for x in entries]
        if len(names) != len(set(names)): raise ValueError(f'Checkpoint có file trùng: {path}')
        if z.testzip() is not None: raise ValueError(f'Checkpoint ZIP hỏng: {path}')
        result = {}
        for item in entries:
            name = item.filename; posix = PurePosixPath(name)
            if not name or name.startswith('/') or '\\' in name or '..' in posix.parts or ':' in name or ((item.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError(f'Đường dẫn checkpoint không an toàn: {name}')
            if not item.is_dir(): result[name] = z.read(item)
        return result

def dir_files(path):
    result = {}
    for p in sorted(path.rglob('*')):
        if p.is_symlink(): raise ValueError(f'Không nhận symlink trong checkpoint: {p}')
        if p.is_file(): result[p.relative_to(path).as_posix()] = p.read_bytes()
    return result

def fingerprint(files):
    # Chỉ nhận cùng một checkpoint nếu toàn bộ record, manifest và khóa giống nhau.
    relevant = {k: hashlib.sha256(v).hexdigest() for k,v in files.items()
                if k.endswith(('/records.jsonl','/run_manifest.json','/thresholds.json','/threshold_lock.json'))}
    if not relevant: raise ValueError('Checkpoint không có record/manifest')
    return relevant

if not RESUME_FROM_CHECKPOINT:
    raise ValueError('Notebook này chỉ resume I-11 hoặc bản mới hơn; phải bật RESUME_FROM_CHECKPOINT')
if CHECKPOINT_INPUT_PATH is not None:
    candidates = [Path(CHECKPOINT_INPUT_PATH)]
else:
    candidates = sorted(p for p in KAGGLE_INPUT_ROOT.rglob('grace_b2_rand_checkpoint.zip') if p.is_file())
    candidates += sorted(p.parent for p in KAGGLE_INPUT_ROOT.rglob('batch_summary.json')
                         if (p.parent/'seed_42').is_dir() and (p.parent/'b2_donor_rerun_66').is_dir())
if not candidates: raise FileNotFoundError('Add Input checkpoint I-11: grace_b2_rand_checkpoint.zip hoặc output đã giải nén.')
loaded=[]
for candidate in candidates:
    if candidate.is_file(): files=zip_files(candidate)
    elif candidate.is_dir(): files=dir_files(candidate)
    else: raise FileNotFoundError(candidate)
    loaded.append((candidate,files,fingerprint(files)))
# Choose a verified descendant rather than file mtime. Older attached Inputs may coexist.
def record_count(files):
    return sum(sum(bool(line.strip()) for line in raw.splitlines()) for name,raw in files.items()
               if name.endswith('/records.jsonl'))
loaded.sort(key=lambda item:record_count(item[1]),reverse=True)
source, checkpoint_files, _ = loaded[0]
for candidate,files,_ in loaded[1:]:
    for name,raw in files.items():
        if name.endswith('/records.jsonl') and not checkpoint_files.get(name,b'').startswith(raw):
            raise ValueError(f'Conflicting checkpoint history: {candidate} vs {source}; choose CHECKPOINT_INPUT_PATH explicitly')
note(f'Chosen checkpoint {source}: {record_count(checkpoint_files)} records including preserved donor history')
OUT_ROOT.mkdir(parents=True, exist_ok=True)
if not any(OUT_ROOT.iterdir()):
    # Ghi từng file có đường dẫn đã kiểm; không sử dụng mã trong checkpoint.
    for name,data in checkpoint_files.items():
        dest=(OUT_ROOT/name).resolve()
        if not dest.is_relative_to(OUT_ROOT.resolve()): raise ValueError(f'Đường dẫn không an toàn: {name}')
        dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
    note(f'Đã khôi phục checkpoint: {source}')
else:
    # Chỉ cho phép thư mục working đã tiến xa hơn cùng checkpoint, không trộn phiên khác.
    for name,data in checkpoint_files.items():
        if not name.endswith('/records.jsonl'): continue
        dest=OUT_ROOT/name
        if not dest.is_file() or not dest.read_bytes().startswith(data):
            raise ValueError(f'Working không chứa đầy đủ record từ checkpoint {source}: {name}; mở session mới')
    note(f'Tiếp tục working directory có cùng lịch sử checkpoint: {OUT_ROOT}')

# Những byte đã chạy I-09 phải còn nguyên ngay cả khi tiếp tục từ checkpoint mới hơn.
PINNED_I09 = {
 'seed_42/devign/validation/records.jsonl': (3575364,'6f6bc0d6f0495fc7e69a834799718768d18ec12a41dfb61e7f2229dfaec91a65'),
 'seed_42/reveal/validation/records.jsonl': (2704748,'74261d5eb005e55513cce0ce1898678e24e83a3cdac843612a7bd9e56b52076e'),
 'b2_donor_rerun_66/devign/validation/records.jsonl': (43764,'794aa82b5a840733985add9daf193ba70d3d6ebbd42d8600dec9343ce1acc20d'),
 'b2_donor_rerun_66/devign/test/records.jsonl': (58899,'082d6f69edbfdb0b5ab38beba130c654aaa1bdae2c117338308a9625d3587d66'),
 'b2_donor_rerun_66/thresholds.json': (12106,'c4273120a64489f38217e73e673afbf2969486248774eb2c0e33cddb6447eb52'),
}
for name,(size,expected_hash) in PINNED_I09.items():
    file=OUT_ROOT/name
    if not file.is_file(): raise ValueError(f'Checkpoint thiếu file I-09: {name}')
    with file.open('rb') as f: prefix=f.read(size)
    if len(prefix)!=size or hashlib.sha256(prefix).hexdigest()!=expected_hash:
        raise ValueError(f'Checkpoint không khớp record I-09: {name}')
    if name.startswith('b2_donor_rerun_66/') and file.stat().st_size != size:
        raise ValueError(f'Output B2 donor đã hoàn tất bị thay đổi: {name}')
note('Đã xác nhận 4.798 record I-09 và output donor; không inference lại donor')

# I-11 is the minimum resumable checkpoint: preserve all 7,789 existing validation rows.
PINNED_I11 = {'seed_42/devign/validation/records.jsonl': (3575364, '6f6bc0d6f0495fc7e69a834799718768d18ec12a41dfb61e7f2229dfaec91a65'), 'seed_42/reveal/validation/records.jsonl': (2975838, '4ecf8514c43d287abce851c717ac458995bd12e1adb7ea15564d484de2fa8315'), 'seed_43/devign/validation/records.jsonl': (3574931, '109e62753f36d25585be977ecefea2cd8a9c513326a271e7279826d00939f44d'), 'seed_43/reveal/validation/records.jsonl': (67980, '5633cd2fc80c8b08580aadcfb7e4847e60ffdb24e9710a36a54cc79602660e3b')}
for name,(size,expected_hash) in PINNED_I11.items():
    path=OUT_ROOT/name
    if not path.is_file(): raise ValueError(f'Checkpoint lacks I-11 data: {name}; attach the latest I-11 checkpoint')
    with path.open('rb') as f: prefix=f.read(size)
    if len(prefix)!=size or hashlib.sha256(prefix).hexdigest()!=expected_hash:
        raise ValueError(f'Checkpoint older than/conflicting with I-11: {name}; attach kaggle_run_02/grace_b2_rand_checkpoint.zip')
note('Verified all 7,789 I-11 validation records; skip completed IDs, resume remaining validation only')
# B2 donor legacy test data is preserved, but B2-rand test records are forbidden.
for seed in SEEDS:
    for dataset in ('devign','reveal'):
        if (OUT_ROOT/f'seed_{seed}'/dataset/'test'/'records.jsonl').exists():
            raise ValueError('B2-rand test records already present; use a separate workflow after threshold locking')


def validate_existing_records():
    progress={}
    for seed in SEEDS:
        for dataset,split in EXPECTED_RUNS:
            key=(seed,dataset,split); item=found[key]; manifest=json.loads(item['manifest'])
            prompt_rows=[json.loads(line) for line in item['prompts'].decode('utf-8').splitlines() if line.strip()]
            expected={str(r['id']):r for r in prompt_rows}
            folder=OUT_ROOT/f'seed_{seed}'/dataset/split; path=folder/'records.jsonl'; run_path=folder/'run_manifest.json'
            if split=='test' and path.exists():
                raise ValueError(f'Checkpoint đã chứa B2-rand test: {key}; dùng notebook phù hợp với phiên đó')
            if not path.exists():
                if run_path.exists(): note(f'Chưa có record: {key}')
                progress[key]=0;continue
            if not run_path.is_file(): raise ValueError(f'Thiếu run_manifest: {key}')
            run=json.loads(run_path.read_text(encoding='utf-8'))
            required={'arm':'B2-rand','prompt_seed':seed,'dataset':dataset,'split':split,'model_sha256':MODEL_SHA256,
                      'tokenizer':manifest['tokenizer'],'tokenizer_revision':manifest['tokenizer_revision'],
                      'context_tokens':manifest['prompt_budget_tokens'],'quantization':manifest['quantization'],
                      'model_repo':manifest['model_repo'],'model_file':MODEL_FILENAME,'llama_cpp_python':'0.3.16',
                      'gpu_layers':GPU_LAYERS,'scoring':'unconstrained_first_token_logits_v1','limit':LIMIT,
                      'model_repo_revision':f'local-sha256:{MODEL_SHA256}','python':platform.python_version(),
                      'prompt_set_sha256':manifest['prompts_sha256']}
            if any(run.get(k)!=v for k,v in required.items()): raise ValueError(f'Run identity không khớp: {key}')
            seen=set()
            with path.open(encoding='utf-8') as stream:
                for line in stream:
                    if not line.strip(): continue
                    row=json.loads(line); uid=str(row['id']); exp=expected.get(uid)
                    if uid in seen or exp is None or (row.get('arm'),row.get('seed'),row.get('dataset'),row.get('split'),row.get('label'),row.get('prompt_sha256'),row.get('model_sha256')) != ('B2-rand',seed,dataset,split,exp['label'],exp['prompt_sha256'],MODEL_SHA256):
                        raise ValueError(f'Record checkpoint không khớp {key}/{uid}')
                    seen.add(uid)
            progress[key]=len(seen)
            note(f'Checkpoint {key}: {len(seen)}/{len(expected)} mẫu; còn {len(expected)-len(seen)}')
    return progress

if LIMIT is not None: raise ValueError('Không dùng LIMIT khi tiếp tục run đầy đủ')
progress=validate_existing_records()
# The deadline was established in setup; do not grant another nine hours here.
RUN_DEADLINE_EPOCH = SESSION_STARTED_EPOCH + MAX_RUNTIME_HOURS * 3600
timed_out=False

state='running'
try:
    for phase in ('validation',):
        for seed in SEEDS:
            needed=[dataset for dataset in ('devign','reveal')
                    if progress[(seed,dataset,phase)] < json.loads(found[(seed,dataset,phase)]['manifest'])['count']]
            if not needed:
                note(f'Bỏ qua seed {seed} {phase}: cả hai split đã đủ và xác minh')
                continue
            if time.time() >= RUN_DEADLINE_EPOCH-CHECKPOINT_RESERVE_SECONDS:
                timed_out=True; break
            status_path=OUT_ROOT/f'seed_{seed}'/f'{phase}_resume_status.json'
            cmd=[sys.executable,'-u','/kaggle/working/kaggle_inference.py','--model',model_path,
                 '--model-revision',revision,'--seed',str(seed),'--arm','B2-rand','--expected-model-sha256',MODEL_SHA256,
                 '--gpu-layers',str(GPU_LAYERS),'--deadline-epoch',str(RUN_DEADLINE_EPOCH),
                 '--checkpoint-reserve-seconds',str(CHECKPOINT_RESERVE_SECONDS),'--status',str(status_path)]
            for dataset in needed:
                cmd += ['--run',dataset,phase,str(BUNDLE_PATHS[(seed,dataset,phase)]),str(OUT_ROOT/f'seed_{seed}'/dataset/phase)]
            run_logged(cmd,f'resume_seed_{seed}_{phase}')
            status=json.loads(status_path.read_text(encoding='utf-8'))
            progress=validate_existing_records()
            if status.get('timed_out'): timed_out=True; break
            if status.get('state') != 'complete' or any(progress[(seed,ds,phase)] != json.loads(found[(seed,ds,phase)]['manifest'])['count'] for ds in needed):
                raise ValueError(f'Inference chưa hoàn tất dù không báo timeout: seed={seed}, phase={phase}')
        if timed_out: break
    if not timed_out and any(progress[(seed,ds,'validation')] != json.loads(found[(seed,ds,'validation')]['manifest'])['count'] for seed in SEEDS for ds in ('devign','reveal')):
        raise ValueError('Chưa đủ validation cả ba seed')
    state='timed_out' if timed_out else 'validation_complete'
except Exception:
    state='error'
    raise
finally:
    summary={'state':state,'timed_out':timed_out,'deadline_epoch':RUN_DEADLINE_EPOCH,'seeds':list(SEEDS),
             'budget_hours':MAX_RUNTIME_HOURS,'session_started_epoch':SESSION_STARTED_EPOCH,'validation_only':True,
             'awaiting_threshold_lock':state=='validation_complete','thresholds_fitted':False,'test_started':False,
             'progress':{f'{seed}/{ds}/{sp}':value for (seed,ds,sp),value in progress.items()}}
    (OUT_ROOT/'batch_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    archive=shutil.make_archive('/kaggle/working/grace_b2_rand_checkpoint','zip',root_dir=OUT_ROOT)
    print(f'State={state}; tải ZIP từ Kaggle Output: {archive}',flush=True)
    diagnostics_zip()


In [ ]:
print('DỪNG: tải grace_b2_rand_checkpoint.zip và grace_diagnostics.zip từ Kaggle Output. Đủ 15.015 validation record thì chốt ngưỡng ở bước riêng; notebook này không fit ngưỡng và không chạy test.')


Không tự chạy test xác nhận trước khi dữ liệu, arm, ngưỡng và protocol được khóa. Nếu model OOM, tokenizer không khớp hoặc thiếu GPU, dừng và lưu log; không thay model hoặc ràng buộc ngầm.
